# Flood Coverage Clustering & Visualisation
Loads the enriched dataset produced by `run_nlp_pipeline.py` and explores actionability distributions across predefined groups (Global North/South, country, source type) and data-driven HDBSCAN clusters.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import importlib

config = importlib.import_module('config.nlp_config')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
FIG_DIR = os.path.join(config.OUTPUT_DIR, 'figures')
os.makedirs(FIG_DIR, exist_ok=True)

def savefig(name):
    path = os.path.join(FIG_DIR, name)
    plt.savefig(path, dpi=150, bbox_inches='tight')
    print(f'saved → {path}')

## 1. Load enriched dataset

In [ ]:
df = pd.read_csv(config.ENRICHED_CSV_PATH)
print(f'{len(df)} articles  |  columns: {list(df.columns)}')
df.head(3)

## 2. Clustering — assign groups + data-driven clusters

In [ ]:
from nlp.clustering import run_clustering
df = run_clustering(df)
print('Columns added:', [c for c in df.columns if c in ('global_region','data_cluster_id','domain')])

## 3. Dataset overview

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

df['language'].value_counts().plot.bar(ax=axes[0], color='steelblue')
axes[0].set_title('Articles by language'); axes[0].set_xlabel('')

df['country'].value_counts().plot.bar(ax=axes[1], color='teal')
axes[1].set_title('Articles by country'); axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=45)

df['global_region'].value_counts().plot.bar(ax=axes[2], color='coral')
axes[2].set_title('Global North vs South'); axes[2].set_xlabel('')

plt.tight_layout()
savefig('01_dataset_overview.png')
plt.show()

## 4. Source type breakdown — the CC bias picture

In [ ]:
if 'source_type' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # overall source type distribution
    st_counts = df['source_type'].value_counts()
    st_counts.plot.bar(ax=axes[0], color='steelblue')
    axes[0].set_title('Source type — overall')
    axes[0].set_xlabel('')
    axes[0].tick_params(axis='x', rotation=30)

    # source type breakdown per country
    ct = df.groupby(['country','source_type']).size().unstack(fill_value=0)
    ct.plot.bar(stacked=True, ax=axes[1], colormap='tab10')
    axes[1].set_title('Source type by country')
    axes[1].set_xlabel('')
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)

    plt.tight_layout()
    savefig('02_source_type_breakdown.png')
    plt.show()
else:
    print('source_type column not found — run run_authority() first')

## 5. Actionability distributions across groups

In [ ]:
score_col = 'actionability_score'

if score_col not in df.columns:
    print(f'{score_col} not found — run actionability step first')
else:
    group_cols = [c for c in ('global_region','language','country','source_type') if c in df.columns]
    fig, axes = plt.subplots(1, len(group_cols), figsize=(5 * len(group_cols), 5))
    if len(group_cols) == 1:
        axes = [axes]

    for ax, col in zip(axes, group_cols):
        order = df.groupby(col)[score_col].median().sort_values(ascending=False).index
        sns.boxplot(data=df, x=col, y=score_col, order=order, ax=ax, palette='muted')
        ax.set_title(f'Actionability by {col}')
        ax.set_xlabel('')
        ax.tick_params(axis='x', rotation=35)

    plt.suptitle('Actionability score distributions', y=1.02, fontsize=13, fontweight='bold')
    plt.tight_layout()
    savefig('03_actionability_distributions.png')
    plt.show()

## 6. Dominant frame by region

In [ ]:
if 'dominant_frame' in df.columns and 'global_region' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # frame distribution: Global North vs South
    fr = df.groupby(['global_region','dominant_frame']).size().unstack(fill_value=0)
    fr_pct = fr.div(fr.sum(axis=1), axis=0) * 100
    fr_pct.plot.bar(stacked=True, ax=axes[0], colormap='Set2')
    axes[0].set_title('Frame distribution — Global North vs South (%)') 
    axes[0].set_xlabel('')
    axes[0].tick_params(axis='x', rotation=0)
    axes[0].yaxis.set_major_formatter(mticker.PercentFormatter())
    axes[0].legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)

    # frame distribution by country
    fc = df.groupby(['country','dominant_frame']).size().unstack(fill_value=0)
    fc_pct = fc.div(fc.sum(axis=1), axis=0) * 100
    fc_pct.plot.bar(stacked=True, ax=axes[1], colormap='Set2')
    axes[1].set_title('Frame distribution by country (%)')
    axes[1].set_xlabel('')
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].yaxis.set_major_formatter(mticker.PercentFormatter())
    axes[1].legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)

    plt.tight_layout()
    savefig('04_frame_by_region.png')
    plt.show()

## 7. Data-driven clusters — what did HDBSCAN find?

In [ ]:
if 'data_cluster_id' not in df.columns:
    print('data_cluster_id not found — clustering may not have run')
else:
    cluster_counts = df['data_cluster_id'].value_counts().sort_index()
    print(f'{(df["data_cluster_id"] >= 0).sum()} articles in {(cluster_counts.index >= 0).sum()} clusters, '
          f'{(df["data_cluster_id"] == -1).sum()} noise')
    print(cluster_counts)

In [ ]:
# feature columns available for cluster profiling
_SCORE_COLS = ['actionability_score','imperative_score','imperative_count',
               'short_term_score','short_term_count','long_term_score','long_term_count',
               'spatial_score','spatial_count','past_tense_ratio']
feat_cols = [c for c in _SCORE_COLS if c in df.columns]
print('Feature columns found:', feat_cols)

In [ ]:
# cluster feature profiles — mean score per cluster
if feat_cols and 'data_cluster_id' in df.columns:
    profile = df.groupby('data_cluster_id')[feat_cols].mean().round(3)
    profile.index = profile.index.map(lambda x: 'noise' if x == -1 else f'cluster_{x}')
    profile.style.background_gradient(axis=0, cmap='YlOrRd')

In [ ]:
# heatmap of cluster feature profiles
if feat_cols and 'data_cluster_id' in df.columns and len(profile) > 1:
    fig, ax = plt.subplots(figsize=(max(8, len(feat_cols) * 1.2), max(4, len(profile) * 0.7)))
    sns.heatmap(profile, annot=True, fmt='.2f', cmap='YlOrRd', linewidths=0.5, ax=ax)
    ax.set_title('Mean feature profile per cluster')
    ax.set_xlabel('')
    plt.tight_layout()
    savefig('05_cluster_feature_profiles.png')
    plt.show()

In [ ]:
# scatter: two most variable features coloured by cluster
if len(feat_cols) >= 2 and 'data_cluster_id' in df.columns:
    variances = df[feat_cols].var().sort_values(ascending=False)
    x_col, y_col = variances.index[0], variances.index[1]

    plot_df = df[df['data_cluster_id'] >= 0].copy()
    noise_df = df[df['data_cluster_id'] == -1].copy()

    fig, ax = plt.subplots(figsize=(8, 6))
    scatter = ax.scatter(plot_df[x_col], plot_df[y_col],
                         c=plot_df['data_cluster_id'], cmap='tab10',
                         alpha=0.7, s=40, label='cluster')
    ax.scatter(noise_df[x_col], noise_df[y_col],
               c='lightgrey', alpha=0.4, s=20, label='noise')
    plt.colorbar(scatter, ax=ax, label='cluster id')
    ax.set_xlabel(x_col); ax.set_ylabel(y_col)
    ax.set_title(f'Data-driven clusters ({x_col} vs {y_col})')
    ax.legend()
    plt.tight_layout()
    savefig('06_cluster_scatter.png')
    plt.show()

## 8. Per-cluster deep dive
For each cluster: dominant country, language, source type, frame, and top domains.

In [ ]:
if 'data_cluster_id' in df.columns:
    for cid in sorted(df['data_cluster_id'].unique()):
        label = 'NOISE' if cid == -1 else f'CLUSTER {cid}'
        subset = df[df['data_cluster_id'] == cid]
        print(f'\n{'='*50}')
        print(f'{label}  —  {len(subset)} articles')
        print(f'{'='*50}')

        if 'actionability_score' in df.columns:
            s = subset['actionability_score']
            print(f'  actionability  mean={s.mean():.3f}  median={s.median():.3f}  std={s.std():.3f}')

        for meta in ('global_region','language','country','source_type','dominant_frame'):
            if meta in subset.columns:
                top = subset[meta].value_counts().head(3)
                print(f'  {meta:20s}: {dict(top)}')

        if 'domain' in subset.columns:
            top_domains = subset['domain'].value_counts().head(5)
            print(f'  top domains        : {dict(top_domains)}')

In [ ]:
# visualise per-cluster composition side by side
if 'data_cluster_id' in df.columns:
    meta_cols = [c for c in ('global_region','source_type','dominant_frame') if c in df.columns]
    n_clusters = df['data_cluster_id'].nunique()

    for meta in meta_cols:
        ct = df.groupby(['data_cluster_id', meta]).size().unstack(fill_value=0)
        ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
        ct_pct.index = ct_pct.index.map(lambda x: 'noise' if x == -1 else f'c{x}')

        ax = ct_pct.plot.bar(stacked=True, figsize=(max(6, n_clusters), 5), colormap='Set2')
        ax.set_title(f'{meta} composition per cluster (%)')
        ax.set_xlabel('cluster')
        ax.yaxis.set_major_formatter(mticker.PercentFormatter())
        ax.tick_params(axis='x', rotation=0)
        ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
        plt.tight_layout()
        savefig(f'07_cluster_composition_{meta}.png')
        plt.show()

## 9. Actionability vs frame — are more actionable articles in the response frame?

In [ ]:
if 'dominant_frame' in df.columns and 'actionability_score' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    frame_order = df.groupby('dominant_frame')['actionability_score'].median().sort_values(ascending=False).index
    sns.boxplot(data=df, x='dominant_frame', y='actionability_score',
                order=frame_order, palette='Set2', ax=axes[0])
    axes[0].set_title('Actionability score by dominant frame')
    axes[0].set_xlabel('')

    if 'global_region' in df.columns:
        sns.boxplot(data=df, x='dominant_frame', y='actionability_score',
                    hue='global_region', order=frame_order, palette='muted', ax=axes[1])
        axes[1].set_title('Actionability by frame × global region')
        axes[1].set_xlabel('')
        axes[1].legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)

    plt.tight_layout()
    savefig('08_actionability_vs_frame.png')
    plt.show()